# Eurostat Energy Balance Evaluation 
Processing of the Eurostat Energy Balance for a specific region/country for the IAMC-variables defined in `/definitions`.

### Data References
- API-url for EB datasource: [https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/nrg_bal_c?format=TSV&compressed=true](https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/nrg_bal_c?format=TSV&compressed=true)
- Definitions of IAMC-Variables: [IIASA/energy-scenarios-at-workflow](https://github.com/iiasa/energy-scenarios-at-workflow/tree/main)

In [ ]:
import pandas as pd 
import pyam 
import nomenclature
import ast


/home/maxnutz/miniconda3/envs/pyam/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [2]:
dsd = nomenclature.DataStructureDefinition("/home/maxnutz/Documents/pypsa-at-validation/validation_pyam_structure/energy_balance_evaluation/definitions")

In [3]:
df_n = dsd.variable.to_pandas()

In [15]:
siec_dict = {
    "Coal": ["C0000X0350-0370", "P1000"],
    "Oil": ["O4000XBIO"],
    "Natural Gas": ["G3000"],
    "Biomass": ["R5110-5150_W6000RI", "R5160", "R5300"],
    "Biogas": ["R5300"],
    "E-Methane": [],
    "Hydro": ["RA100"],
    "Solar": ["RA420"],
    "Wind": ["RA300"],
    "Geothermal": ["RA200"],
    "Waste": ["W6100_6220"],
    "Other Renewables": ["RA500"],
    "Electricity": ["E7000"],
    "Biofuels": ["R5210P", "R5220P", "R5230P", "R5290", "R5160"],
    "E-Fuels": [None],
    "Biomethane": ["R5300"],
    "Hydrogen": [None],
    "Solar Thermal": ["RA410"],
    "District Heat": ["H8000"],
    "Ambient Heat": ["RA600"],
    "Other": [None],
    "Renewables": [
        "RA100", "RA500", "RA300", "RA420", "RA410", "RA200",
        "R5110-5150_W6000RI", "R5160", "R5300", "W6210",
        "R5210P", "R5210B", "R5220P", "R5220B", "R5230P", "R5230B", "R5290",
    ],
    "Renewables incl. Ambient Heat": ["RA000"],
}

nrg_dict = {
    "Net Imports|{Final Energy Carrier}": ["IMP", "-EXP"],
    "Final Energy": ["FC_E"],
    "Final Energy [by Carrier]|{Final Energy Carrier}": ["FC_E"],
    "Final Energy [by Sector]|Agriculture": ["FC_OTH_AF_E"],
    "Final Energy [by Sector]|Agriculture|{Final Energy Carrier}": ["FC_OTH_AF_E"],
    "Final Energy [by Sector]|Industry": ["FC_IND_E"],
    "Final Energy [by Sector]|Industry|{Final Energy Carrier}": ["FC_IND_E"],
    "Final Energy [by Sector]|Residential and Commercial": ["FC_OTH_CP_E", "FC_OTH_HH_E"],
    "Final Energy [by Sector]|Residential and Commercial|{Final Energy Carrier}": ["FC_OTH_CP_E", "FC_OTH_HH_E"],
    "Final Energy [by Sector]|Transportation": [
        "FC_TRA_RAIL_E", "FC_TRA_ROAD_E", "FC_TRA_DAVI_E", "FC_TRA_DNAVI_E", "FC_TRA_NSP_E",
    ],
    "Final Energy [by Sector]|Transportation|{Final Energy Carrier}": [
        "FC_TRA_RAIL_E", "FC_TRA_ROAD_E", "FC_TRA_DAVI_E", "FC_TRA_DNAVI_E", "FC_TRA_NSP_E",
    ],
    "Final Energy [by Sector]|{Sector}": [],
    "Final Energy [by Sector]|{Sector}|{Final Energy Carrier}": [],
    "Secondary Energy|Electricity": ["GEP"],
    "Secondary Energy|Electricity|{Electricity Generation by Source}": [
        "TI_EHG_MAPE_E", "TI_EHG_MAPCHP_E", "TI_EHG_APE_E", "TI_EHG_APCHP_E",
    ],
}

no direct mapping of variables possible, but multiindex-structure to read out variables of EB to be mapped on the IAMC-Variables.

In [5]:
from energy_balance_eval import fetch_and_load_tsv_data
eb_df = fetch_and_load_tsv_data("resources/pypsa-eur-download.tsv")
eb_df = eb_df[eb_df.geo == "AT"]


In [6]:
eb_df

,freq,nrg_bal,siec,unit,geo,1990,1991,1992,1993,1994,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,A,AFC,BIOE,GWH,AT,25744.633,27905.111,27110.141,27367.890,25186.310,...,57495.297,57863.539,58217.651,55736.679,55563.334,54830.084,60953.408,59869.408,62017.419,62384.826
124,A,AFC,C0000X0350-0370,GWH,AT,12661.389,13590.083,11523.667,10434.278,9353.833,...,4341.902,4376.606,4261.576,3897.165,3900.608,4261.570,3252.060,3163.126,2890.301,2991.250
247,A,AFC,C0110,GWH,AT,124.444,54.444,85.556,23.333,15.556,...,34.658,515.859,594.705,494.412,173.003,171.127,221.651,20.480,10.790,15.245
370,A,AFC,C0121,GWH,AT,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.008,0.000,0.000
493,A,AFC,C0129,GWH,AT,2994.444,3266.667,3118.889,3243.333,2831.111,...,1473.825,1253.676,880.001,945.711,919.631,1280.268,1027.206,695.620,263.160,399.124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1215895,A,TO_RPI_RO,TOTAL,GWH,AT,104779.056,109969.472,112574.944,113330.056,115384.819,...,107073.577,100221.588,99436.287,107616.280,111732.628,99737.125,98577.594,72428.991,96783.957,98867.475
1216018,A,TO_RPI_RO,W6100,GWH,AT,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216138,A,TO_RPI_RO,W6100_6220,GWH,AT,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216258,A,TO_RPI_RO,W6210,GWH,AT,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_n

,variable,description,unit,EB,note
0,Final Energy,Final energy consumption by all end-use sector...,TJ,FC_E,Final energy consumption explicitly excludes e...
1,Secondary Energy|Electricity,Total net generation of electricity (including...,TWh,GEP,NaN
2,Final Energy [by Carrier]|Coal,Energy consumption of coal excluding transmiss...,TJ,FC_E,Final energy consumption explicitly excludes e...
3,Final Energy [by Carrier]|Oil,Energy consumption of oil excluding transmissi...,TJ,FC_E,Final energy consumption explicitly excludes e...
4,Final Energy [by Carrier]|Natural Gas,Energy consumption of fossil methane excluding...,TJ,FC_E,Final energy consumption explicitly excludes e...
...,...,...,...,...,...
121,Net Imports|District Heat,Net imports of heat delivered from district he...,"[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.
122,Net Imports|Ambient Heat,"Net imports of ambient heat using heat pumps, ...","[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.
123,Net Imports|Other,Net imports of other sources or carriers,"[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.
124,Net Imports|Renewables,Net imports of renewables including biomass an...,"[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.


### IAMC variable calculation from Eurostat energy balance

This cell computes IAMC-style variable values by combining `eb_df` with the variable definitions in `df_n` using the mapping dictionaries `siec_dict` and `nrg_dict`.

- **Input data**:
  - `eb_df` (Eurostat energy-balance data, already filtered to a region, e.g. `AT`)
  - `df_n` (variable definition table from the nomenclature DSD)
  - `siec_dict` (maps IAMC carrier/source names to Eurostat `siec` codes)
  - `nrg_dict` (maps IAMC variable templates to Eurostat `nrg_bal` codes)

- **How matching works**:
  - Each variable name from `df_n` is matched against template keys in `nrg_dict` (including placeholders like `{Final Energy Carrier}` or `{Electricity Generation by Source}`).
  - Placeholder values are resolved via `siec_dict` to obtain one or more `siec` codes.
  - The selected `nrg_bal` rows are aggregated over all detected year columns.
  - Signed entries in `nrg_dict` (e.g. `-EXP`) are subtracted.

- **Missing/incomplete mappings**:
  - If no template match exists in `nrg_dict`, the result is `TOTAL` to cover aggregated values
  - If a matched `nrg_dict` entry is empty, the result is `np.nan`.
  - If a required `siec_dict` entry is empty/missing, the result is `np.nan`.
  - If no matching rows are found in `eb_df`, the result is `np.nan`.

- **Unit Conversion**
  - values are read in in GWh
  - values are converted to the respective values defined in `/definitions`


- **Outputs**:
  - `calculated_values_df`: computed values per IAMC variable and year.
  - `df_n_with_values`: original `df_n` plus the computed year columns.
  - `_df`: pyam.IamDataFrame with output values for the computed year(s). Column `note` is not included.

In [46]:
import re
import numpy as np
import pandas as pd

year_cols: list[str] = ["2024"] # [col for col in eb_df.columns if re.fullmatch(r"\d{4}", str(col))] # define a single year as list holding one entry.


if not year_cols:
    raise ValueError("No year columns found in eb_df.")

def _get_variable_list(df_variables: pd.DataFrame) -> list[str]:
    if "variable" in df_variables.columns:
        return df_variables["variable"].astype(str).tolist()
    else:
        raise ValueError("Dataset holding variable definitions is not of valid nomenclature.DataStructureDefinition.variable - structure.")

def _match_nrg_template(variable_name: str, mapping: dict[str, list[str]]) -> tuple[str | None, dict[str, str]]:
    for template in mapping:
        pattern = "^" + re.escape(template) + "$"
        placeholders = re.findall(r"\{([^}]+)\}", template)
        for placeholder in placeholders:
            pattern = pattern.replace(r"\{" + re.escape(placeholder) + r"\}", r"(.+?)", 1)

        match = re.match(pattern, variable_name)
        if match:
            values = {placeholder: match.group(i + 1) for i, placeholder in enumerate(placeholders)}
            return template, values

    return None, {}

def _get_siec_codes(placeholder_values: dict[str, str]) -> list[str] | None:
    siec_codes: list[str] = []

    for placeholder, value in placeholder_values.items():
        if placeholder in {"Final Energy Carrier", "Electricity Generation by Source"}:
            codes = siec_dict.get(value, None)
            if codes is None:
                continue
            if codes == [None]:
                return None
            if not codes:
                continue
            siec_codes.extend([code for code in codes if code is not None])

    return sorted(set(siec_codes))

def _calculate_single_variable(variable_name: str) -> pd.Series:
    log_variable_code = [variable_name]
    template, placeholder_values = _match_nrg_template(variable_name, nrg_dict)
    if template is None:
        return pd.Series(np.nan, index=year_cols, dtype=float), log_variable_code

    nrg_codes = nrg_dict.get(template, [])
    if not nrg_codes:
        return pd.Series(np.nan, index=year_cols, dtype=float), log_variable_code

    siec_codes = _get_siec_codes(placeholder_values)
    if siec_codes is None:
        return pd.Series(np.nan, index=year_cols, dtype=float), log_variable_code
    if not siec_codes:
        siec_codes = ["TOTAL"]

    result = pd.Series(0.0, index=year_cols, dtype=float)
    has_data = False

    log_variable_code.append(["nrg_codes: ", ", ".join(nrg_codes)])
    log_variable_code.append(["siec_codes: ", ", ".join(siec_codes)])

    for code in nrg_codes:
        sign = -1.0 if str(code).startswith("-") else 1.0
        clean_code = str(code).lstrip("-")

        subset = eb_df.loc[eb_df["nrg_bal"] == clean_code].copy()
        if siec_codes:
            subset = subset.loc[subset["siec"].isin(siec_codes)]

        if subset.empty:
            continue

        values = subset[year_cols].apply(pd.to_numeric, errors="coerce").sum(axis=0, min_count=1)
        result = result.add(sign * values, fill_value=np.nan)
        has_data = True

    if not has_data:
        return pd.Series(np.nan, index=year_cols, dtype=float)

    return result, log_variable_code

variable_list = _get_variable_list(df_n)

variable_definition_log: list[str] = []
calculated_records: list[dict[str, object]] = []
for variable in variable_list:
    value_series, log = _calculate_single_variable(variable)
    variable_definition_log.append(log)
    record = {"variable": variable, **value_series.to_dict()}
    calculated_records.append(record)

calculated_values_df = pd.DataFrame(calculated_records).set_index("variable")

df_n_with_values = df_n.merge(calculated_values_df, on="variable", how="left")
df_n_with_values["unit"] = "GWh"
df_n_with_values.head()

,variable,description,unit,EB,note,2024
0,Final Energy,Final energy consumption by all end-use sector...,GWh,FC_E,Final energy consumption explicitly excludes e...,279335.902
1,Secondary Energy|Electricity,Total net generation of electricity (including...,GWh,GEP,NaN,82406.732
2,Final Energy [by Carrier]|Coal,Energy consumption of coal excluding transmiss...,GWh,FC_E,Final energy consumption explicitly excludes e...,2597.812
3,Final Energy [by Carrier]|Oil,Energy consumption of oil excluding transmissi...,GWh,FC_E,Final energy consumption explicitly excludes e...,88969.655
4,Final Energy [by Carrier]|Natural Gas,Energy consumption of fossil methane excluding...,GWh,FC_E,Final energy consumption explicitly excludes e...,43842.553


In [49]:
year = 2024
df_n_with_values.rename(columns = {"variable": "variable_name", "unit": "unit_EB" }, inplace = True)
_df = pyam.IamDataFrame(
    data = df_n_with_values.drop_duplicates().drop(columns = ["EB", "note"]), # empty note-cells lead to error.
    model=f"Eurostat Energy Balance {year}",
    scenario="historical data",
    region = "AT",
    variable = "variable_name",
    unit = "unit_EB"
)

In [ ]:
dsd_variable_df = dsd.variable.to_pandas().copy()

def _first_unit(unit_value: object) -> str | None:
    if unit_value is None:
        return None

    if isinstance(unit_value, str):
        stripped = unit_value.strip()
        if not stripped:
            return None
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(stripped)
                if isinstance(parsed, list):
                    if not parsed:
                        return None
                    first = parsed[0]
                    return None if pd.isna(first) else str(first)
            except (SyntaxError, ValueError):
                pass
        return stripped

    if isinstance(unit_value, list):
        if not unit_value:
            return None
        first = unit_value[0]
        return None if pd.isna(first) else str(first)

    if pd.isna(unit_value):
        return None

    return str(unit_value)

target_unit_map: dict[str, str] = {}
for _, row in dsd_variable_df[["variable", "unit"]].drop_duplicates(subset=["variable"]).iterrows():
    target_unit = _first_unit(row["unit"])
    if target_unit is not None:
        target_unit_map[str(row["variable"])] = target_unit

converted_df = _df
pairs_to_check = converted_df.as_pandas()[["variable", "unit"]].drop_duplicates()

successful_conversions: list[tuple[str, str, str]] = []
failed_conversions: list[tuple[str, str, str]] = []

for _, pair in pairs_to_check.iterrows():
    variable_name = str(pair["variable"])
    current_unit = str(pair["unit"])
    target_unit = target_unit_map.get(variable_name)

    if target_unit is None or current_unit == target_unit:
        continue

    subset = converted_df.filter(variable=variable_name, unit=current_unit)
    if subset.empty:
        continue

    try:
        subset_converted = subset.convert_unit(
            current=current_unit,
            to=target_unit,
            inplace=False,
        )
        remainder = converted_df.filter(variable=variable_name, unit=current_unit, keep=False)
        converted_df = pyam.concat([remainder, subset_converted])
        successful_conversions.append((variable_name, current_unit, target_unit))
    except Exception:
        failed_conversions.append((variable_name, current_unit, target_unit))

_df = converted_df

if failed_conversions:
    print(f"Successful conversions: {len(successful_conversions)}")
    print(f"Failed conversions: {len(failed_conversions)}")

In [68]:
_df.to_excel("resources/iamc_eb_2024.xlsx")